In [ ]:
import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_v3.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)
ragas_df

In [5]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory(
    "gpt-4o-mini",
    provider="openai",
    client=async_client,
    max_tokens=4096,
    max_retries=6,
    timeout=120,  
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy,
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1516


statement: 蘇彥碩在使用Git時遇到的最大問題是，有人不會使用Git。
verdict: 1
reason: The context explicitly mentions that one of the major problems is that some people do not know how to use Git, which directly relates to 蘇彥碩's experience.


Evaluating:  10%|▉         | 19/200 [00:10<01:13,  2.45it/s]


statement: 在選擇SDGs主題時，需遵循以下原則。
verdict: 1
reason: The context outlines principles for selecting topics, including the SDGs, which supports this statement.

statement: 題目須排除過去三年的題目。
verdict: 1
reason: The context explicitly states that topics must exclude those from the past three years.

statement: 題目選擇應依據TronClass公告為準。
verdict: 1
reason: The context mentions that topic selection should be based on TronClass announcements.

statement: 應盡量選擇可以找到實際使用者進行訪談的題目。
verdict: 1
reason: The context advises to choose topics that allow for interviews with actual users.

statement: 根據SDGs，應挑選一個有興趣的SDG。
verdict: 1
reason: The context instructs to select an interesting SDG based on the SDGs.

statement: 從挑選的SDG中尋找主題。
verdict: 1
reason: The context indicates that one should find a topic from the selected SDG.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  14%|█▍        | 28/200 [00:14<00:57,  2.97it/s]

...


statement: 根據朱浩偉的經驗，朱浩偉在專題製作中建議要安排一個計畫。
verdict: 1
reason: 朱浩偉的經驗中提到要安排計畫以確保小組進度，因此這個陳述可以直接推斷。

statement: 安排計畫可以確保小組進度能夠跟著計劃進行。
verdict: 1
reason: 根據朱浩偉的經驗，安排計畫的確是為了確保小組進度，因此這個陳述是正確的。

statement: 朱浩偉建議把握每個可以寫小專案的機會。
verdict: 1
reason: 朱浩偉的經驗中提到要把握每一個可以寫小專案的機會，因此這個陳述是正確的。

statement: 釐清系統開發對使用者的價值會讓專題製作更順利。
verdict: 1
reason: 朱浩偉提到釐清系統開發對使用者的價值會使專題製作更順利，因此這個陳述是正確的。

statement: 至於時間投入方面，初期每週可能需要10-15小時。
verdict: 1
reason: 朱浩偉提到初期每週可能需要10-15小時的時間投入，因此這個陳述是正確的。

statement: 初期的時間主要用於研究和規劃。
verdict: 1
reason: 根據朱浩偉的經驗，初期的時間確實主要用於研究和規劃，因此這個陳述是正確的。

statement: 在進入開發和整合階段，接近死線或與產學合作方有審查時，每週投入的時間會增加到20-30小時甚至更多。
verdict: 1
reason: 朱浩偉提到在開發和整合階段，時間投入會增加到20-30小時甚至更多，因此這個陳述是正確的。


Evaluating: 100%|█████████▉| 199/200 [01:54<00:01,  1.16s/it]


statement: 林芯緹和蘇彥碩建議在共同開發時要事先溝通好程式的命名規則。
verdict: 1
reason: 根據林芯緹、蘇彥碩的經驗，確實提到要溝通好程式之命名規則，因此這個陳述可以直接推斷。

statement: 程式的命名規則最好以功能來命名。
verdict: 1
reason: 學姐建議以功能命名，這與陳述一致，因此可以直接推斷。

statement: 這樣可以讓所有人以及未來的自己都清楚程式碼的意義。
verdict: 1
reason: 文中提到可以將自己的邏輯想法寫在註解，讓大家和未來的自己都清楚程式碼的意義，因此這個陳述可以直接推斷。

statement: 另外，建議在開發過程中定期舉行「debug 大會」來追蹤組員的進度。
verdict: 1
reason: 文中提到偶爾開個debug 大會，這與陳述一致，因此可以直接推斷。

statement: 在「debug 大會」中，應該適時關心每位組員的狀況。
verdict: 1
reason: 文中提到並適時關心每一個組員進度，這與陳述一致，因此可以直接推斷。

statement: 這樣可以確保大家都在同一頁。
verdict: 1
reason: 雖然文中沒有明確提到這一點，但關心組員的狀況有助於確保大家在同一頁，因此可以推斷出這個陳述的合理性。

statement: 提高整體合作的效率是最終目標。
verdict: 1
reason: 文中提到要分工合作才能完成整個Project，這暗示了提高合作效率的重要性，因此可以推斷出這個陳述的合理性。


Evaluating: 100%|██████████| 200/200 [01:55<00:00,  1.73it/s]


statement: 根據蔡易辰的經驗，使用MySQL的優點是因為上學期（大二上）才剛學過。
verdict: 1
reason: 蔡易辰提到使用MySQL的原因是因為上學期剛學過，因此這個陳述可以直接推斷。

statement: 對於學生來說，使用MySQL比較熟悉。
verdict: 0
reason: 雖然蔡易辰提到使用MySQL的原因，但並未明確指出學生對MySQL的熟悉程度，因此無法直接推斷。

statement: 學生可以快速上手使用MySQL。
verdict: 0
reason: 蔡易辰提到上學期剛學過MySQL，但並未提到學生能否快速上手，因此無法直接推斷。

statement: 蔡易辰強烈建議不要只用MySQL這個資料庫。
verdict: 1
reason: 蔡易辰在上下文中明確提到不建議只用MySQL，因此這個陳述可以直接推斷。

statement: 在SA階段還有更多時間學習其他技術。
verdict: 1
reason: 蔡易辰提到在SA階段有更多時間學習其他程式語言，因此這個陳述可以直接推斷。

statement: 蔡易辰並未明確提到MySQL的缺點。
verdict: 1
reason: 蔡易辰在上下文中並未提到MySQL的缺點，因此這個陳述可以直接推斷。

statement: 如果專題要用另一個語言開發，可能會需要重新學習。
verdict: 1
reason: 蔡易辰提到如果專題要用另一個語言開發，會需要花時間從頭學習，因此這個陳述可以直接推斷。

statement: 這意味著若選擇MySQL而未掌握其他資料庫，轉換時可能會影響專題的進度。
verdict: 0
reason: 雖然蔡易辰提到需要學習其他技術，但並未明確提到選擇MySQL會影響專題進度，因此無法直接推斷。

statement: 目前沒有彭文暘的相關經驗資料。
verdict: 1
reason: 上下文中並未提到彭文暘的經驗，因此這個陳述可以直接推斷。

statement: 無法提供彭文暘的觀點。
verdict: 1
reason: 由於上下文中沒有提到彭文暘的觀點，因此這個陱述可以直接推斷。
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9

In [3]:
from pathlib import Path
import json
import ast
from datetime import datetime

import pandas as pd
import openai

from dotenv import load_dotenv

from ragas import evaluate, EvaluationDataset
from ragas.metrics import (
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
from ragas.llms import llm_factory
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()


# =========================
# 1. 設定資料路徑
# =========================

TARGET_RUN_DIR = Path("eval_ragas_testset/20260603/v1")

INPUT_CSV = TARGET_RUN_DIR / "ragas_eval_with_response.csv"

# 輸出檔案
OUTPUT_SCORES_CSV = TARGET_RUN_DIR / "ragas_scores.csv"
OUTPUT_SUMMARY_MD = TARGET_RUN_DIR / "ragas_summary.md"
OUTPUT_SUMMARY_JSON = TARGET_RUN_DIR / "ragas_summary.json"


# =========================
# 2. 讀取 CSV
# =========================

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
# df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig", nrows=1)

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# RAGAS 需要的欄位
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# =========================
# 3. 設定 LLM 與 Embedding
# =========================

async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory(
    "gpt-4o-mini",
    provider="openai",
    client=async_client,
    max_tokens=4096,
    max_retries=6,
    timeout=120,
)

ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(
    llm=evaluator_llm,
    embeddings=ragas_embeddings,
)

answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""


# =========================
# 4. 執行 RAGAS 評估
# =========================

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy,
    ],
)

print(scores)


# =========================
# 5. 將結果轉成 DataFrame
# =========================

scores_df = scores.to_pandas()

# 加入原始問題與回答，方便之後檢查
result_df = pd.concat(
    [
        df.reset_index(drop=True),
        scores_df.reset_index(drop=True),
    ],
    axis=1,
)

# 輸出每題分數 CSV
result_df.to_csv(OUTPUT_SCORES_CSV, index=False, encoding="utf-8-sig")


# =========================
# 6. 計算平均分數
# =========================

metric_columns = [
    col
    for col in scores_df.columns
    if col not in ["user_input", "retrieved_contexts", "response", "reference"]
]

average_scores = scores_df[metric_columns].mean(numeric_only=True).to_dict()

question_count = len(df)

run_info = {
    "run_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target_run_dir": str(TARGET_RUN_DIR),
    "input_csv": str(INPUT_CSV),
    "question_count": question_count,
    "average_scores": average_scores,
    "output_scores_csv": str(OUTPUT_SCORES_CSV),
    "output_summary_md": str(OUTPUT_SUMMARY_MD),
}


# =========================
# 7. 輸出 JSON 摘要
# =========================

with open(OUTPUT_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)


# =========================
# 8. 輸出好讀的 Markdown 摘要
# =========================

md_lines = []

md_lines.append("# RAGAS 評估結果摘要")
md_lines.append("")
md_lines.append(f"執行時間：`{run_info['run_time']}`")
md_lines.append("")
md_lines.append("## 資料來源")
md_lines.append("")
md_lines.append(f"- 評估資料夾：`{TARGET_RUN_DIR}`")
md_lines.append(f"- 輸入檔案：`{INPUT_CSV}`")
md_lines.append(f"- 題數：`{question_count}` 題")
md_lines.append("")
md_lines.append("## 平均分數")
md_lines.append("")

for metric_name, score_value in average_scores.items():
    md_lines.append(f"- `{metric_name}`：{score_value:.4f}")

md_lines.append("")
md_lines.append("## 輸出檔案")
md_lines.append("")
md_lines.append(f"- 每題詳細分數：`{OUTPUT_SCORES_CSV}`")
md_lines.append(f"- 摘要 JSON：`{OUTPUT_SUMMARY_JSON}`")
md_lines.append(f"- 摘要 Markdown：`{OUTPUT_SUMMARY_MD}`")
md_lines.append("")


with open(OUTPUT_SUMMARY_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))


print("評估完成！")
print(f"每題詳細分數已輸出：{OUTPUT_SCORES_CSV}")
print(f"摘要 Markdown 已輸出：{OUTPUT_SUMMARY_MD}")
print(f"摘要 JSON 已輸出：{OUTPUT_SUMMARY_JSON}")

C:\Users\User\AppData\Local\Temp\ipykernel_14480\399669551.py:12: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_14480\399669551.py:12: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_14480\399669551.py:12: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_14480\39966955


statement: 吳建豐在設計使用者故事時遇到的問題是對「user story」這個概念不熟悉。
verdict: 1
reason: 根據上下文，吳建豐剛開始對user story這個概念不熟悉，這確實是他在設計使用者故事時遇到的問題。

statement: 吳建豐感到對「user story」這個概念非常陌生。
verdict: 1
reason: 上下文中提到吳建豐剛開始學到user story這個概念時感到非常陌生，因此這個陳述是正確的。

statement: 這種陌生感導致吳建豐無法一次就把user story的表格設定好。
verdict: 1
reason: 上下文中提到因為不熟悉user story的概念，吳建豐無法一次設定好表格，因此這個陳述是正確的。

statement: 隨著課程的進行，吳建豐跟著老師的指導。
verdict: 1
reason: 上下文中提到吳建豐隨著課程的進行，跟著老師的腳步去思考，因此這個陳述是正確的。

statement: 吳建豐開始站在使用者的立場思考。
verdict: 1
reason: 上下文中提到吳建豐跟著老師的指導，能夠站在使用者的立場思考，因此這個陳述是正確的。

statement: 因此，吳建豐能逐漸適應並進行多次調整。
verdict: 1
reason: 上下文中提到隨著課程的進行，吳建豐能夠進行多次調整，因此這個陳述是正確的。


Evaluating:  12%|█▏        | 24/200 [00:10<00:52,  3.34it/s]


statement: 根據蘇彥碩的經驗，蘇彥碩建議能儘早學習使用 Git。
verdict: 1
reason: 蘇彥碩的經驗中提到建議儘早學習使用 Git，因此這個陳述可以直接推斷。

statement: 蘇彥碩特別建議在大家都還不熟悉的時候，可以設定 pull requests。
verdict: 1
reason: 根據蘇彥碩的經驗，他建議在大家不熟悉 Git 時設定 pull requests，因此這個陳述可以直接推斷。

statement: 設定 pull requests 可以讓團隊確保程式碼無誤後再進行最後的 merge。
verdict: 1
reason: 根據林芯緹的經驗，設定 pull requests 的目的是為了確保程式碼無誤再進行 merge，因此這個陳述可以直接推斷。

statement: 蘇彥碩建議要學習使用終端機下指令。
verdict: 1
reason: 根據蘇彥碩的經驗，他提到學習使用終端機下指令是必要的，因此這個陳述可以直接推斷。

statement: 學習使用終端機下指令包括切換路徑等操作。
verdict: 1
reason: 根據上下文，學習使用終端機下指令確實包括切換路徑等操作，因此這個陳述可以直接推斷。

statement: 這樣的做法可以幫助團隊有效地協作。
verdict: 0
reason: 雖然上下文提到的做法有助於團隊協作，但並未明確指出這一點，因此無法直接推斷。

statement: 這樣的做法可以減少錯誤發生的機會。
verdict: 0
reason: 上下文中提到的做法可能有助於減少錯誤，但並未明確指出，因此無法直接推斷。


Evaluating:  15%|█▌        | 30/200 [00:11<00:28,  5.89it/s]

...


statement: 彭文暘在系統開發中遇到的問題主要是資料庫的規劃不完整。
verdict: 1
reason: The context explicitly states that the database was not well planned, which led to issues in connecting it.

statement: 資料庫無法連結是由於規劃不完整所導致的。
verdict: 1
reason: The context mentions that the database could not connect due to incomplete planning, making this statement directly inferable.

statement: 彭文暘提到，當資料庫每次更動後，需要將其壓縮上傳到群組。
verdict: 1
reason: The context states that every time the database is changed, it is compressed and uploaded to the group, confirming this statement.

statement: 壓縮上傳的目的是讓組員得到最新的資料庫。
verdict: 1
reason: The context explains that compressing and uploading the database allows team members to have the latest version, thus this statement is valid.

statement: 彭文暘表示在組內有共識後，問題就能很好解決。
verdict: 1
reason: The context indicates that once there is consensus within the group, problems can be resolved effectively, supporting this statement.

statement: 使用D hub的優點是因為之前學過D hub。
verdict: 1
reason: The context mentions that

Evaluating: 100%|█████████▉| 199/200 [01:38<00:01,  1.92s/it]


statement: 在「系統分析與設計」課程中，利用 TronClass 平台進行主題提案報告的準備，可以遵循以下步驟。
verdict: 0
reason: 雖然上下文提到 TronClass 平台用於課程公告和作業提交，但並未具體列出準備主題提案報告的步驟，因此無法直接推斷。

statement: TronClass 平台會發布課程的相關公告。
verdict: 1
reason: 上下文明確提到 TronClass 平台用於課程公告，因此這一陳述可以直接推斷。

statement: 使用者應先查看是否有關於主題提案報告的指引或要求。
verdict: 1
reason: 上下文中提到學生應主動探索和查看課程相關資源，這暗示了查看指引或要求的重要性，因此可以推斷。

statement: 利用 TronClass 的討論區或小組功能，進行組員之間的溝通與分工。
verdict: 0
reason: 上下文提到 TronClass 平台的公告和作業提交，但未提及討論區或小組功能，因此無法直接推斷。

statement: 確保每位組員了解報告的內容與要點。
verdict: 0
reason: 上下文強調了組員之間的溝通與協調，但並未具體提到確保每位組員了解報告內容，因此無法直接推斷。

statement: 根據報告要求整理文件。
verdict: 1
reason: 上下文提到報告內容須包含特定要素，這暗示了需要根據報告要求整理文件，因此可以推斷。

statement: 整理的文件應包含問題陳述、利害關係人分析、需求蒐集方法、競爭者分析及市場定位等。
verdict: 1
reason: 上下文中明確列出了報告需包含的內容，因此這一陳述可以直接推斷。

statement: 完成報告後，根據 TronClass 上的指示，進行作業的提交。
verdict: 1
reason: 上下文提到作業提交需依據 TronClass 上的資訊，因此這一陳述可以直接推斷。

statement: 確保不錯過截止時間。
verdict: 1
reason: 上下文中提到作業提交的截止時間，這暗示了需要注意截止時間，因此可以推斷。

statement: 這些步驟可以幫助使用者更有效地利用 TronClass 平台進行主題提案報告的準備與執行。
verdict: 0


Evaluating: 100%|██████████| 200/200 [01:40<00:00,  1.98it/s]


statement: 在第二章中，使用者角色的描述對於清楚定義系統需求非常重要。
verdict: 1
reason: 第二章強調使用者角色的描述是系統需求的關鍵部分，因此這一陳述可以直接推斷。

statement: 使用者角色的描述能幫助團隊理解系統將為哪些使用者服務。
verdict: 1
reason: 文中提到使用者角色的描述應該清楚說明系統有哪些使用者，因此這一陳述是正確的。

statement: 使用者角色的描述能幫助團隊理解各使用者在系統中的主要使用情境。
verdict: 1
reason: 文中提到使用者角色的描述應包括主要使用情境，因此這一陳述是正確的。

statement: 這樣的理解能確保需求的適切性與針對性。
verdict: 1
reason: 文中提到清楚的使用者角色描述能夠確保需求的適切性，因此這一陳述是正確的。

statement: 這樣的理解能避免開發過程中出現的角色不一致或需求描述混淆的問題。
verdict: 1
reason: 文中提到清晰的使用者角色描述能避免角色不一致的問題，因此這一陳述是正確的。

statement: 透過清晰的使用者角色描述，開發者能夠準確聚焦於使用者的實際需求。
verdict: 1
reason: 文中提到清晰的使用者角色描述能幫助開發者聚焦於需求，因此這一陳述是正確的。

statement: 清晰的使用者角色描述能最終提高系統的使用效益與用戶滿意度。
verdict: 1
reason: 文中提到清晰的使用者角色描述能提高系統的使用效益，因此這一陳述是正確的。

statement: 如果使用者角色未能清楚定義，可能會導致與 User Story 中的角色不一致的問題。
verdict: 1
reason: 文中提到未清楚定義使用者角色會導致角色不一致，因此這一陳述是正確的。

statement: 如果使用者角色未能清楚定義，可能會導致未能清楚說明使用者角色與使用情境的問題。
verdict: 1
reason: 文中提到未清楚定義使用者角色會導致說明不清，因此這一陳述是正確的。

statement: 明確定義使用者角色是系統需求規範的基礎。
verdict: 1
reason: 文中強調明確定義使用者角色是需求規範的基礎，因此這一陳述是正確的。

statement: 明確定

In [30]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory("gpt-4o-mini", provider="openai", client=async_client,max_tokens=4096)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1195

{'faithfulness': 0.7917, 'context_precision': 0.9615, 'context_recall': 0.9038, 'answer_relevancy': 0.9091}


ragas_eval_with_response_v3
ragas_testset_v2
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

3.1
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

"ragas_eval_with_response_2026-05-22.csv"

{
    "faithfulness": 0.7917,
    "context_precision": 0.9615,
    "context_recall": 0.9038,
    "answer_relevancy": 0.9091,
}

ragas_eval_with_response_v3
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9333, 'answer_relevancy': 0.7727}